# 05. Dropout의 학습 모드와 평가 모드

2026-10-30 실습 기록 · 원본: `../05_dropout.py`


## 0. 작업 전 준비사항
PyTorch가 설치된 Python 커널을 선택하고 **Run All(모두 실행)**로 위에서 아래로 실행합니다. 작은 CPU 텐서만 사용하며 데이터 파일이나 GPU는 필요하지 않습니다.

## 1. 실습 목적과 입력
같은 입력을 Dropout에 반복해서 넣고 학습 모드와 평가 모드의 출력을 비교합니다.

- 입력: `torch.ones(2, 4)` — 모든 원소가 1인 2행 4열 텐서
- 제거 확률: `p=0.5`
- 난수 시드: `42`

이 실습은 Dropout의 동작을 관찰합니다. 모델 학습이나 Optimizer에 의한 가중치 갱신은 수행하지 않습니다.


In [1]:
import torch
from torch import nn

print("PyTorch 버전:", torch.__version__)
torch.manual_seed(42)
X = torch.ones(2, 4)
dropout = nn.Dropout(p=0.5)


PyTorch 버전: 2.14.0+cu130


## 2. 학습 모드에서 세 번 실행
`dropout.train()`으로 Dropout을 활성화합니다. 각 forward 호출에서 원소별로 새로운 마스크를 뽑습니다. 마스크가 우연히 같을 수도 있습니다.

제거된 원소는 0이 되고, 유지된 원소는 `1 / (1 - p)`배로 조정됩니다. 이번에는 `1 / (1 - 0.5) = 2`이므로 출력이 0 또는 2입니다. 이 조정은 출력의 기댓값을 유지하기 위한 것입니다. 각 호출에서 정확히 절반이 0이 된다는 뜻은 아닙니다.


In [2]:
dropout.train()
for i in range(3):
    print("train", i + 1, dropout(X).tolist())


train 1 [[2.0, 2.0, 2.0, 2.0], [0.0, 2.0, 0.0, 0.0]]
train 2 [[2.0, 2.0, 2.0, 2.0], [0.0, 0.0, 2.0, 0.0]]
train 3 [[2.0, 0.0, 0.0, 2.0], [2.0, 0.0, 2.0, 2.0]]


## 3. B06 작성: 평가 모드로 전환
실습에서 작성한 코드는 `dropout.eval()`입니다.

`dropout.eval()`, `dropout.train(False)`, `dropout.train(mode=False)`는 동일하게 해당 모듈을 평가 모드로 전환합니다.

평가 모드에서는 Dropout이 입력을 그대로 전달하므로 모든 출력이 1입니다. 학습 모드에서 이미 스케일을 조정했으므로 평가 시 추가로 배율을 곱하지 않습니다.


In [3]:
dropout.eval()  # dropout.train(mode=False)와 동일
for i in range(3):
    print("eval", i + 1, dropout(X).tolist())


eval 1 [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]
eval 2 [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]
eval 3 [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]


## 4. 결과 해석
### 완료 질문: train은 왜 0 또는 2이고, eval은 왜 1인가?
학습 모드에서는 각 원소를 확률 0.5로 제거하고 남은 값을 2배로 조정합니다. 평가 모드에서는 제거와 배율 조정 없이 입력을 그대로 전달하므로 1입니다. 고정된 시드로 실행해도 PyTorch 버전이나 실행 환경에 따라 구체적인 마스크는 다를 수 있습니다.

### Dropout은 다음 batch로 전달할지를 제어하는가?
Dropout은 **현재 forward에서 어떤 활성값을 다음 층으로 전달할지** 제어합니다. 이전 batch의 값이나 기울기를 다음 batch로 넘길지 결정하는 기능은 아닙니다.

가중치 자체를 삭제하지 않으며, 역전파에서는 해당 forward의 마스크를 그대로 사용합니다. 제거된 경로를 통한 기울기는 0이고, 유지된 경로에는 학습 시 적용한 배율이 반영됩니다. 공유 가중치에는 다른 경로의 기울기가 기여할 수 있고, 모멘텀이나 weight decay 등도 있으므로 특정 가중치가 반드시 갱신되지 않는다고 일반화하지 않습니다.

### eval()과 no_grad()의 차이
| 호출 | 역할 |
| --- | --- |
| `dropout.eval()` | 해당 Dropout을 평가 모드로 전환 |
| `model.eval()` | 모델과 모든 하위 모듈을 평가 모드로 전환. Dropout과 BatchNorm 등의 동작에 영향 |
| `torch.no_grad()` | 문맥 안에서 수행하는 연산의 기울기 기록을 비활성화. 모드를 바꾸지는 않음 |

**`eval()`만 호출해도 기울기 계산은 가능합니다.** 아래에서 입력에 대한 기울기로 확인합니다. 일반적인 추론에서는 `model.eval()`과 `torch.no_grad()`를 함께 사용합니다.


In [4]:
probe = torch.ones(2, 4, requires_grad=True)
dropout.eval()
output = dropout(probe)
output.sum().backward()
print("eval에서 입력 기울기:", probe.grad.tolist())

with torch.no_grad():
    result = dropout(probe) * 2
    print("no_grad 안의 연산 결과 requires_grad:", result.requires_grad)
print("Dropout training:", dropout.training)


eval에서 입력 기울기: [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]
no_grad 안의 연산 결과 requires_grad: False
Dropout training: False


### 자주 하는 실수 점검
- 학습 모드에서 활성값을 제거하는 것과 가중치 자체를 삭제하는 것을 구분합니다.
- `eval()`은 Dropout을 비활성화하며 기울기 기록을 끄는 기능과는 별개입니다.
- 평가 후 다시 학습한다면 `model.train()`으로 학습 모드로 되돌립니다.
- 이번 실습의 Dropout은 일반 `nn.Dropout`으로, 뉴런 전체를 batch 공통으로 끄는 것이 아니라 입력 원소별 마스크를 사용합니다.
